# Notebook 08 v2 — Predicción conformal multiclase canónica

## Grain AI–MDPI

Esta versión reemplaza el análisis conformal anterior y utiliza tres scores claramente definidos:

- **LAC:** \(1-\hat p_y\).
- **APS aleatorizado:** masa acumulada de las clases anteriores más \(U\hat p_y\).
- **RAPS aleatorizado:** APS más una penalización de regularización.

La aleatorización es reproducible y se realiza con **un único valor uniforme por grupo** (`duplicate_group_id`), compartido por todas las clases candidatas de la misma unidad. Los cuantiles se ajustan únicamente con calibración y la evaluación principal se realiza a nivel de grupo, que es la unidad de intercambio usada para prevenir fuga por duplicados.

### Análisis preespecificado

- Modelos campeones congelados del Notebook 04.
- Probabilidades nativas y escaladas por temperatura.
- Calibración marginal y Mondrian por clase.
- Coberturas nominales de 90% y 95%.
- LAC, APS y RAPS.
- Bootstrap estratificado por clase y agrupado.
- Comparaciones pareadas:
  - APS frente a LAC.
  - RAPS frente a APS.
  - Mondrian frente a marginal para APS.

No se entrena ningún modelo ni se selecciona un método mediante la prueba.

In [ ]:
# ============================================================
# 1. Dependencias, configuración y directorios
# ============================================================
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pyarrow": "pyarrow",
    "statsmodels": "statsmodels",
}
for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

RANDOM_SEED = 20260805
RANDOMIZATION_SEED = 2026080501
N_BOOTSTRAP = 2000
CONFIDENCE_LEVEL = 0.95

DATASETS = ["dry_bean", "iniap_rice"]
PROBABILITY_STATUSES = ["native", "temperature_scaled"]
NOMINAL_COVERAGES = [0.90, 0.95]
CALIBRATION_MODES = ["marginal", "mondrian"]

CONFORMAL_METHODS = {
    "LAC": {"lambda_reg": 0.0, "k_reg": 1},
    "APS": {"lambda_reg": 0.0, "k_reg": 1},
    "RAPS": {"lambda_reg": 0.01, "k_reg": 1},
}

PRIMARY_PROBABILITY_STATUS = "temperature_scaled"
PRIMARY_METHOD = "APS"
PRIMARY_CALIBRATION_MODE = "marginal"

RUNTIME_ROOT = (
    Path("/content")
    if Path("/content").exists()
    else Path("/mnt/data")
)
LOCAL_ROOT = RUNTIME_ROOT / "Grain_AI_MDPI_notebook08_v2"
INPUT_DIR = LOCAL_ROOT / "00_input"
EXTRACT04_DIR = LOCAL_ROOT / "01_extracted_notebook04"
EXTRACT05_DIR = LOCAL_ROOT / "02_extracted_notebook05"
EXTRACT07_DIR = LOCAL_ROOT / "03_extracted_notebook07"
OUTPUT_ROOT = LOCAL_ROOT / "04_outputs"

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

PROTOCOL_DIR = OUTPUT_ROOT / "03_protocol"
RESULTS_DIR = OUTPUT_ROOT / "07_results" / "08_V2_CANONICAL_CONFORMAL"
FIGURES_DIR = OUTPUT_ROOT / "08_figures" / "08_V2_CANONICAL_CONFORMAL"
TABLES_DIR = OUTPUT_ROOT / "09_tables" / "08_V2_CANONICAL_CONFORMAL"

for directory in [
    INPUT_DIR,
    EXTRACT04_DIR,
    EXTRACT05_DIR,
    EXTRACT07_DIR,
    PROTOCOL_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED_SCENARIOS = (
    len(DATASETS)
    * len(PROBABILITY_STATUSES)
    * len(CONFORMAL_METHODS)
    * len(CALIBRATION_MODES)
    * len(NOMINAL_COVERAGES)
)

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("Bootstrap:", N_BOOTSTRAP)
print("Escenarios esperados:", EXPECTED_SCENARIOS)

In [ ]:
# ============================================================
# 2. Google Drive, extracción y archivos de entrada
# ============================================================
try:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    files = None
    print("No se detectó Google Colab; se usarán rutas locales.")

DRIVE_PROJECT_ROOT = (
    Path("/content/drive/MyDrive/Grain_Robustness_Q1")
    if IN_COLAB
    else Path("/mnt/data")
)
DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

ZIP_FILENAMES = {
    "notebook04": "NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip",
    "notebook05": "NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip",
    "notebook07": "NOTEBOOK07_SELECTIVE_UNCERTAINTY_OUTPUTS.zip",
}

def find_exact_file(roots, filename):
    matches = []
    for root in roots:
        root = Path(root)
        if root.exists():
            matches.extend(root.rglob(filename))
    if not matches:
        return None
    return max(matches, key=lambda path: path.stat().st_mtime)

def obtain_zip(filename):
    search_roots = [
        DRIVE_PROJECT_ROOT,
        Path("/content"),
        Path("/mnt/data"),
    ]
    found = find_exact_file(search_roots, filename)
    if found is not None:
        print("Encontrado:", found)
        return found

    if IN_COLAB and files is not None:
        print("Suba:", filename)
        uploaded = files.upload()
        if filename not in uploaded:
            raise FileNotFoundError(
                f"Debe subir exactamente {filename}."
            )
        destination = INPUT_DIR / filename
        destination.write_bytes(uploaded[filename])
        return destination

    raise FileNotFoundError(filename)

def clean_extract(zip_path, destination):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(destination)

zip_paths = {
    key: obtain_zip(filename)
    for key, filename in ZIP_FILENAMES.items()
}

clean_extract(zip_paths["notebook04"], EXTRACT04_DIR)
clean_extract(zip_paths["notebook05"], EXTRACT05_DIR)
clean_extract(zip_paths["notebook07"], EXTRACT07_DIR)

SELECTED_CONFIG_FILE = (
    EXTRACT04_DIR
    / "07_results"
    / "04_MODEL_SELECTION"
    / "03_selected_model_configurations.csv"
)
PREDICTION_ROOT = (
    EXTRACT05_DIR
    / "06_predictions"
    / "05_BASELINE"
)
NOTEBOOK07_VALIDATION_FILE = (
    EXTRACT07_DIR
    / "07_results"
    / "07_SELECTIVE_UNCERTAINTY"
    / "05_final_selective_validation.csv"
)

for required_path in [
    SELECTED_CONFIG_FILE,
    PREDICTION_ROOT,
    NOTEBOOK07_VALIDATION_FILE,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

selected_configurations = pd.read_csv(SELECTED_CONFIG_FILE)
notebook07_validation = pd.read_csv(NOTEBOOK07_VALIDATION_FILE)

if not notebook07_validation["passed"].astype(bool).all():
    display(notebook07_validation)
    raise RuntimeError("El Notebook 07 no superó sus validaciones.")

print("Entradas extraídas y Notebook 07 validado.")

In [ ]:
# ============================================================
# 3. Campeones, predicciones y auditoría
# ============================================================
training_champions = (
    selected_configurations.sort_values(
        ["dataset_id", "macro_f1_mean", "log_loss_mean", "candidate_id"],
        ascending=[True, False, True, True],
    )
    .groupby("dataset_id", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

if len(training_champions) != len(DATASETS):
    raise RuntimeError("Número inesperado de campeones.")

def load_class_map(path):
    mapping = json.loads(path.read_text(encoding="utf-8"))
    return [str(mapping[str(index)]) for index in range(len(mapping))]

def load_prediction(dataset_id, representation, algorithm, split):
    directory = PREDICTION_ROOT / dataset_id / representation
    prediction_path = directory / f"{algorithm}_{split}_predictions.parquet"
    class_map_path = directory / f"{algorithm}_class_map.json"

    if not prediction_path.exists() or not class_map_path.exists():
        raise FileNotFoundError(
            f"Falta {prediction_path} o {class_map_path}."
        )

    frame = pd.read_parquet(prediction_path).copy()
    class_order = load_class_map(class_map_path)

    required_columns = {
        "record_id",
        "duplicate_group_id",
        "class_label",
        "partition",
    }
    missing_columns = sorted(required_columns - set(frame.columns))
    if missing_columns:
        raise ValueError(
            f"Faltan columnas en {prediction_path}: {missing_columns}"
        )

    frame["record_id"] = frame["record_id"].astype(str)
    frame["duplicate_group_id"] = frame["duplicate_group_id"].astype(str)
    frame["class_label"] = frame["class_label"].astype(str)
    frame = frame.sort_values("record_id").reset_index(drop=True)

    expected_partition = (
        "calibration" if split == "calibration" else "test"
    )
    if not (frame["partition"] == expected_partition).all():
        raise ValueError(f"Partición inesperada en {prediction_path}.")

    native_columns = [
        f"prob_native_{index}"
        for index in range(len(class_order))
    ]
    calibrated_columns = [
        f"prob_calibrated_{index}"
        for index in range(len(class_order))
    ]

    for columns in [native_columns, calibrated_columns]:
        missing_probability_columns = [
            column for column in columns if column not in frame.columns
        ]
        if missing_probability_columns:
            raise ValueError(
                f"Faltan probabilidades: {missing_probability_columns}"
            )
        probabilities = frame[columns].to_numpy(dtype=float)
        if not np.isfinite(probabilities).all():
            raise ValueError(f"Probabilidades no finitas: {prediction_path}")
        if not np.allclose(
            probabilities.sum(axis=1),
            1.0,
            atol=1e-8,
        ):
            raise ValueError(
                f"Probabilidades no normalizadas: {prediction_path}"
            )

    return {
        "frame": frame,
        "class_order": class_order,
        "native_probability_columns": native_columns,
        "calibrated_probability_columns": calibrated_columns,
        "prediction_path": prediction_path,
    }

predictions = {}
audit_rows = []

for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]

    for split in ["calibration", "test"]:
        key = (dataset_id, representation, algorithm, split)
        predictions[key] = load_prediction(*key)

    calibration_bundle = predictions[
        (dataset_id, representation, algorithm, "calibration")
    ]
    test_bundle = predictions[
        (dataset_id, representation, algorithm, "test")
    ]

    calibration_frame = calibration_bundle["frame"]
    test_frame = test_bundle["frame"]

    group_overlap = set(
        calibration_frame["duplicate_group_id"]
    ).intersection(set(test_frame["duplicate_group_id"]))
    record_overlap = set(
        calibration_frame["record_id"]
    ).intersection(set(test_frame["record_id"]))

    calibration_group_labels = calibration_frame.groupby(
        "duplicate_group_id"
    )["class_label"].nunique()
    test_group_labels = test_frame.groupby(
        "duplicate_group_id"
    )["class_label"].nunique()

    audit_rows.append(
        {
            "dataset_id": dataset_id,
            "representation": representation,
            "algorithm": algorithm,
            "calibration_rows": len(calibration_frame),
            "test_rows": len(test_frame),
            "calibration_groups": calibration_frame[
                "duplicate_group_id"
            ].nunique(),
            "test_groups": test_frame[
                "duplicate_group_id"
            ].nunique(),
            "group_overlap": len(group_overlap),
            "record_overlap": len(record_overlap),
            "same_class_order": (
                calibration_bundle["class_order"]
                == test_bundle["class_order"]
            ),
            "calibration_groups_multiple_labels": int(
                (calibration_group_labels > 1).sum()
            ),
            "test_groups_multiple_labels": int(
                (test_group_labels > 1).sum()
            ),
        }
    )

input_audit = pd.DataFrame(audit_rows)

invalid_input = (
    (input_audit["group_overlap"] > 0)
    | (input_audit["record_overlap"] > 0)
    | (~input_audit["same_class_order"])
    | (input_audit["calibration_groups_multiple_labels"] > 0)
    | (input_audit["test_groups_multiple_labels"] > 0)
)

if invalid_input.any():
    display(input_audit)
    raise RuntimeError("Falló la auditoría de entradas.")

input_audit.to_csv(
    RESULTS_DIR / "01_prediction_input_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    training_champions[
        ["dataset_id", "representation", "algorithm", "candidate_id"]
    ]
)
display(input_audit)

## Definiciones conformales

Para una clase candidata \(y\):

- **LAC:** \(S_{\mathrm{LAC}}(x,y)=1-\hat p_y(x)\).
- **APS:** \(S_{\mathrm{APS}}(x,y,U)=\sum_{j<r(y)}\hat p_{(j)}(x)+U\hat p_y(x)\).
- **RAPS:** \(S_{\mathrm{RAPS}}=S_{\mathrm{APS}}+\lambda\max(r(y)-k_{\mathrm{reg}},0)\).

El mismo \(U\) se utiliza para todas las clases candidatas de un grupo. Los valores uniformes se generan mediante SHA-256 a partir de una semilla fija, el conjunto, la partición y el identificador de grupo.

In [ ]:
# ============================================================
# 4. Funciones conformales canónicas y métricas
# ============================================================
def probability_matrix(bundle, status):
    if status == "native":
        columns = bundle["native_probability_columns"]
    elif status == "temperature_scaled":
        columns = bundle["calibrated_probability_columns"]
    else:
        raise ValueError(status)

    return bundle["frame"][columns].to_numpy(dtype=float)

def encode_labels(labels, class_order):
    mapping = {
        class_label: index
        for index, class_label in enumerate(class_order)
    }
    return np.array(
        [mapping[str(label)] for label in labels],
        dtype=np.int64,
    )

def stable_uniform(group_id, namespace, seed):
    payload = f"{seed}|{namespace}|{group_id}".encode("utf-8")
    digest = hashlib.sha256(payload).digest()
    integer_value = int.from_bytes(digest[:8], byteorder="big")
    uniform_value = (integer_value + 0.5) / (2**64)
    return float(np.clip(uniform_value, 1e-12, 1.0 - 1e-12))

def group_uniforms(frame, namespace, seed):
    group_ids = frame["duplicate_group_id"].astype(str)
    mapping = {
        group_id: stable_uniform(group_id, namespace, seed)
        for group_id in sorted(group_ids.unique())
    }
    return group_ids.map(mapping).to_numpy(dtype=float)

def candidate_score_matrix(
    probabilities,
    method_name,
    random_uniforms,
    lambda_reg=0.0,
    k_reg=1,
):
    probabilities = np.asarray(probabilities, dtype=float)
    random_uniforms = np.asarray(random_uniforms, dtype=float)

    if probabilities.ndim != 2:
        raise ValueError("Las probabilidades deben ser una matriz.")
    if len(random_uniforms) != len(probabilities):
        raise ValueError("Longitud incorrecta de uniformes.")

    if method_name == "LAC":
        return 1.0 - probabilities

    if method_name not in {"APS", "RAPS"}:
        raise ValueError(method_name)

    n_rows, n_classes = probabilities.shape
    sorted_indices = np.argsort(
        -probabilities,
        axis=1,
        kind="mergesort",
    )
    sorted_probabilities = np.take_along_axis(
        probabilities,
        sorted_indices,
        axis=1,
    )
    cumulative_before = (
        np.cumsum(sorted_probabilities, axis=1)
        - sorted_probabilities
    )

    randomized_mass = (
        random_uniforms.reshape(-1, 1)
        * sorted_probabilities
    )

    if method_name == "APS":
        penalties = np.zeros(n_classes, dtype=float)
    else:
        ranks = np.arange(1, n_classes + 1, dtype=float)
        penalties = float(lambda_reg) * np.maximum(
            ranks - float(k_reg),
            0.0,
        )

    sorted_scores = (
        cumulative_before
        + randomized_mass
        + penalties.reshape(1, -1)
    )

    score_matrix = np.empty_like(sorted_scores)
    np.put_along_axis(
        score_matrix,
        sorted_indices,
        sorted_scores,
        axis=1,
    )
    return score_matrix

def true_class_scores(score_matrix, true_indices):
    return score_matrix[
        np.arange(len(true_indices)),
        true_indices,
    ]

def group_calibration_table(frame, true_scores):
    working = pd.DataFrame(
        {
            "duplicate_group_id": (
                frame["duplicate_group_id"].astype(str).to_numpy()
            ),
            "class_label": (
                frame["class_label"].astype(str).to_numpy()
            ),
            "true_score": np.asarray(true_scores, dtype=float),
        }
    )

    group_table = (
        working.groupby("duplicate_group_id", as_index=False)
        .agg(
            class_label=("class_label", "first"),
            class_count=("class_label", "nunique"),
            group_score=("true_score", "max"),
            group_rows=("true_score", "size"),
        )
    )

    if (group_table["class_count"] != 1).any():
        raise RuntimeError(
            "Un grupo de calibración contiene varias clases."
        )

    return group_table

def conformal_quantile(scores, alpha):
    scores = np.asarray(scores, dtype=float)
    scores = scores[np.isfinite(scores)]
    n_units = len(scores)

    if n_units == 0:
        raise ValueError("No hay scores de calibración.")

    order_index = int(
        np.ceil((n_units + 1) * (1.0 - float(alpha)))
    )
    order_index = min(max(order_index, 1), n_units)
    ordered_scores = np.sort(scores)

    return {
        "qhat": float(ordered_scores[order_index - 1]),
        "n_calibration_units": int(n_units),
        "order_index": int(order_index),
        "empirical_quantile_level": float(
            order_index / n_units
        ),
    }

def build_quantiles(
    group_table,
    class_order,
    alpha,
    calibration_mode,
):
    if calibration_mode == "marginal":
        result = conformal_quantile(
            group_table["group_score"].to_numpy(),
            alpha,
        )
        qhat_by_class = {
            class_label: result["qhat"]
            for class_label in class_order
        }
        details = pd.DataFrame(
            [{"class_label": "__MARGINAL__", **result}]
        )
        return qhat_by_class, details

    if calibration_mode == "mondrian":
        rows = []
        qhat_by_class = {}

        for class_label in class_order:
            class_scores = group_table.loc[
                group_table["class_label"] == class_label,
                "group_score",
            ].to_numpy()

            result = conformal_quantile(
                class_scores,
                alpha,
            )
            qhat_by_class[class_label] = result["qhat"]
            rows.append({"class_label": class_label, **result})

        return qhat_by_class, pd.DataFrame(rows)

    raise ValueError(calibration_mode)

def prediction_sets_from_scores(
    score_matrix,
    class_order,
    qhat_by_class,
):
    thresholds = np.array(
        [qhat_by_class[class_label] for class_label in class_order],
        dtype=float,
    )
    return score_matrix <= thresholds.reshape(1, -1)

def validate_group_consistency(
    frame,
    prediction_sets,
):
    encoded_sets = [
        "".join("1" if value else "0" for value in row)
        for row in prediction_sets
    ]
    working = pd.DataFrame(
        {
            "duplicate_group_id": (
                frame["duplicate_group_id"].astype(str).to_numpy()
            ),
            "encoded_set": encoded_sets,
        }
    )
    inconsistent_groups = (
        working.groupby("duplicate_group_id")["encoded_set"]
        .nunique()
        .gt(1)
        .sum()
    )
    return int(inconsistent_groups)

def group_representative_indices(frame):
    return (
        frame.reset_index()
        .drop_duplicates("duplicate_group_id", keep="first")["index"]
        .to_numpy(dtype=np.int64)
    )

def evaluate_prediction_sets(
    frame,
    prediction_sets,
    class_order,
    evaluation_unit,
):
    if evaluation_unit == "group":
        selected_indices = group_representative_indices(frame)
    elif evaluation_unit == "row":
        selected_indices = np.arange(len(frame), dtype=np.int64)
    else:
        raise ValueError(evaluation_unit)

    selected_frame = frame.iloc[selected_indices].reset_index(drop=True)
    selected_sets = prediction_sets[selected_indices]

    true_labels = selected_frame[
        "class_label"
    ].astype(str).to_numpy()
    true_indices = encode_labels(true_labels, class_order)
    row_indices = np.arange(len(selected_frame))

    covered = selected_sets[row_indices, true_indices]
    set_sizes = selected_sets.sum(axis=1).astype(int)

    class_rows = []
    for class_index, class_label in enumerate(class_order):
        class_mask = true_indices == class_index
        class_rows.append(
            {
                "class_label": class_label,
                "support": int(class_mask.sum()),
                "coverage": float(covered[class_mask].mean()),
                "mean_set_size": float(
                    set_sizes[class_mask].mean()
                ),
                "singleton_rate": float(
                    np.mean(set_sizes[class_mask] == 1)
                ),
            }
        )

    class_table = pd.DataFrame(class_rows)

    return {
        "evaluation_unit": evaluation_unit,
        "selected_indices": selected_indices,
        "covered": covered,
        "set_sizes": set_sizes,
        "true_indices": true_indices,
        "coverage": float(covered.mean()),
        "mean_set_size": float(set_sizes.mean()),
        "median_set_size": float(np.median(set_sizes)),
        "singleton_rate": float(np.mean(set_sizes == 1)),
        "empty_set_rate": float(np.mean(set_sizes == 0)),
        "full_set_rate": float(
            np.mean(set_sizes == len(class_order))
        ),
        "minimum_class_coverage": float(
            class_table["coverage"].min()
        ),
        "maximum_class_coverage": float(
            class_table["coverage"].max()
        ),
        "class_coverage_range": float(
            class_table["coverage"].max()
            - class_table["coverage"].min()
        ),
        "n_evaluation_units": int(len(selected_frame)),
        "class_table": class_table,
        "selected_frame": selected_frame,
    }

def percentile_interval(values, confidence_level=0.95):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan

    lower_probability = (1.0 - confidence_level) / 2.0
    upper_probability = 1.0 - lower_probability
    return (
        float(np.quantile(values, lower_probability)),
        float(np.quantile(values, upper_probability)),
    )

def bootstrap_two_sided_pvalue(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n_values = len(values)

    probability_nonpositive = (
        np.count_nonzero(values <= 0) + 1
    ) / (n_values + 1)
    probability_nonnegative = (
        np.count_nonzero(values >= 0) + 1
    ) / (n_values + 1)

    return float(
        min(
            1.0,
            2.0
            * min(
                probability_nonpositive,
                probability_nonnegative,
            ),
        )
    )

In [ ]:
# ============================================================
# 5. Calibración y evaluación de los 48 escenarios
# ============================================================
quantile_tables = []
global_rows = []
classwise_tables = []
set_size_rows = []
scenario_evaluations = {}
score_diagnostic_rows = []
all_set_sizes_valid = True

for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]

    calibration_bundle = predictions[
        (dataset_id, representation, algorithm, "calibration")
    ]
    test_bundle = predictions[
        (dataset_id, representation, algorithm, "test")
    ]

    calibration_frame = calibration_bundle["frame"]
    test_frame = test_bundle["frame"]
    class_order = calibration_bundle["class_order"]

    calibration_true_indices = encode_labels(
        calibration_frame["class_label"].to_numpy(),
        class_order,
    )

    calibration_uniforms = group_uniforms(
        calibration_frame,
        namespace=f"{dataset_id}|calibration",
        seed=RANDOMIZATION_SEED,
    )
    test_uniforms = group_uniforms(
        test_frame,
        namespace=f"{dataset_id}|test",
        seed=RANDOMIZATION_SEED,
    )

    for probability_status in PROBABILITY_STATUSES:
        calibration_probabilities = probability_matrix(
            calibration_bundle,
            probability_status,
        )
        test_probabilities = probability_matrix(
            test_bundle,
            probability_status,
        )

        for method_name, method_config in CONFORMAL_METHODS.items():
            calibration_score_matrix = candidate_score_matrix(
                probabilities=calibration_probabilities,
                method_name=method_name,
                random_uniforms=calibration_uniforms,
                lambda_reg=method_config["lambda_reg"],
                k_reg=method_config["k_reg"],
            )
            test_score_matrix = candidate_score_matrix(
                probabilities=test_probabilities,
                method_name=method_name,
                random_uniforms=test_uniforms,
                lambda_reg=method_config["lambda_reg"],
                k_reg=method_config["k_reg"],
            )

            calibration_true_scores = true_class_scores(
                calibration_score_matrix,
                calibration_true_indices,
            )
            group_table = group_calibration_table(
                calibration_frame,
                calibration_true_scores,
            )

            top_indices = calibration_probabilities.argmax(axis=1)
            top_scores = calibration_score_matrix[
                np.arange(len(calibration_frame)),
                top_indices,
            ]
            score_diagnostic_rows.append(
                {
                    "dataset_id": dataset_id,
                    "probability_status": probability_status,
                    "method": method_name,
                    "mean_top_class_score": float(top_scores.mean()),
                    "minimum_top_class_score": float(top_scores.min()),
                    "maximum_top_class_score": float(top_scores.max()),
                    "fraction_top_scores_exactly_zero": float(
                        np.mean(np.isclose(top_scores, 0.0))
                    ),
                }
            )

            for nominal_coverage in NOMINAL_COVERAGES:
                alpha = 1.0 - nominal_coverage

                for calibration_mode in CALIBRATION_MODES:
                    qhat_by_class, quantile_details = build_quantiles(
                        group_table=group_table,
                        class_order=class_order,
                        alpha=alpha,
                        calibration_mode=calibration_mode,
                    )

                    quantile_details = quantile_details.assign(
                        dataset_id=dataset_id,
                        representation=representation,
                        algorithm=algorithm,
                        probability_status=probability_status,
                        method=method_name,
                        calibration_mode=calibration_mode,
                        nominal_coverage=nominal_coverage,
                        lambda_reg=method_config["lambda_reg"],
                        k_reg=method_config["k_reg"],
                        randomization_seed=RANDOMIZATION_SEED,
                    )
                    quantile_tables.append(quantile_details)

                    prediction_sets = prediction_sets_from_scores(
                        score_matrix=test_score_matrix,
                        class_order=class_order,
                        qhat_by_class=qhat_by_class,
                    )

                    set_sizes_all_rows = prediction_sets.sum(axis=1)
                    all_set_sizes_valid = bool(
                        all_set_sizes_valid
                        and (set_sizes_all_rows >= 0).all()
                        and (set_sizes_all_rows <= len(class_order)).all()
                    )

                    inconsistent_groups = validate_group_consistency(
                        test_frame,
                        prediction_sets,
                    )
                    if inconsistent_groups > 0:
                        raise RuntimeError(
                            f"{inconsistent_groups} grupos tienen "
                            "conjuntos de predicción inconsistentes."
                        )

                    scenario_key = (
                        dataset_id,
                        probability_status,
                        method_name,
                        calibration_mode,
                        nominal_coverage,
                    )

                    for evaluation_unit in ["group", "row"]:
                        evaluation = evaluate_prediction_sets(
                            frame=test_frame,
                            prediction_sets=prediction_sets,
                            class_order=class_order,
                            evaluation_unit=evaluation_unit,
                        )

                        global_rows.append(
                            {
                                "dataset_id": dataset_id,
                                "representation": representation,
                                "algorithm": algorithm,
                                "probability_status": probability_status,
                                "method": method_name,
                                "calibration_mode": calibration_mode,
                                "nominal_coverage": nominal_coverage,
                                "evaluation_unit": evaluation_unit,
                                "coverage": evaluation["coverage"],
                                "coverage_gap": (
                                    evaluation["coverage"]
                                    - nominal_coverage
                                ),
                                "mean_set_size": evaluation[
                                    "mean_set_size"
                                ],
                                "median_set_size": evaluation[
                                    "median_set_size"
                                ],
                                "singleton_rate": evaluation[
                                    "singleton_rate"
                                ],
                                "empty_set_rate": evaluation[
                                    "empty_set_rate"
                                ],
                                "full_set_rate": evaluation[
                                    "full_set_rate"
                                ],
                                "minimum_class_coverage": evaluation[
                                    "minimum_class_coverage"
                                ],
                                "maximum_class_coverage": evaluation[
                                    "maximum_class_coverage"
                                ],
                                "class_coverage_range": evaluation[
                                    "class_coverage_range"
                                ],
                                "n_evaluation_units": evaluation[
                                    "n_evaluation_units"
                                ],
                            }
                        )

                        class_table = evaluation["class_table"].copy()
                        class_table = class_table.assign(
                            dataset_id=dataset_id,
                            representation=representation,
                            algorithm=algorithm,
                            probability_status=probability_status,
                            method=method_name,
                            calibration_mode=calibration_mode,
                            nominal_coverage=nominal_coverage,
                            evaluation_unit=evaluation_unit,
                        )
                        classwise_tables.append(class_table)

                        set_size_counts = (
                            pd.Series(evaluation["set_sizes"])
                            .value_counts()
                            .sort_index()
                        )
                        for set_size, count in set_size_counts.items():
                            set_size_rows.append(
                                {
                                    "dataset_id": dataset_id,
                                    "probability_status": probability_status,
                                    "method": method_name,
                                    "calibration_mode": calibration_mode,
                                    "nominal_coverage": nominal_coverage,
                                    "evaluation_unit": evaluation_unit,
                                    "set_size": int(set_size),
                                    "count": int(count),
                                    "proportion": float(
                                        count
                                        / evaluation["n_evaluation_units"]
                                    ),
                                }
                            )

                        if evaluation_unit == "group":
                            scenario_evaluations[scenario_key] = {
                                "covered": evaluation["covered"],
                                "set_sizes": evaluation["set_sizes"],
                                "true_indices": evaluation["true_indices"],
                                "class_order": class_order,
                                "selected_frame": evaluation[
                                    "selected_frame"
                                ],
                                "representation": representation,
                                "algorithm": algorithm,
                            }

conformal_quantiles = pd.concat(
    quantile_tables,
    ignore_index=True,
)
global_results = pd.DataFrame(global_rows)
classwise_results = pd.concat(
    classwise_tables,
    ignore_index=True,
)
set_size_distribution = pd.DataFrame(set_size_rows)
score_diagnostics = pd.DataFrame(score_diagnostic_rows)

if len(scenario_evaluations) != EXPECTED_SCENARIOS:
    raise RuntimeError(
        "Escenarios conformales principales incompletos: "
        f"{len(scenario_evaluations)} de {EXPECTED_SCENARIOS}."
    )

expected_global_rows = EXPECTED_SCENARIOS * 2
if len(global_results) != expected_global_rows:
    raise RuntimeError(
        f"Filas globales: {len(global_results)}; "
        f"esperadas: {expected_global_rows}."
    )

aps_diagnostics = score_diagnostics[
    score_diagnostics["method"] == "APS"
]
if (
    aps_diagnostics["fraction_top_scores_exactly_zero"]
    >= 0.999
).any():
    display(aps_diagnostics)
    raise RuntimeError(
        "APS sigue degenerando a scores superiores exactamente cero."
    )

conformal_quantiles.to_csv(
    TABLES_DIR / "Table_conformal_calibration_quantiles_v2.csv",
    index=False,
    encoding="utf-8-sig",
)
global_results.to_csv(
    TABLES_DIR / "Table_conformal_global_test_results_v2.csv",
    index=False,
    encoding="utf-8-sig",
)
classwise_results.to_csv(
    TABLES_DIR / "Table_conformal_classwise_test_results_v2.csv",
    index=False,
    encoding="utf-8-sig",
)
set_size_distribution.to_csv(
    TABLES_DIR / "Table_conformal_set_size_distribution_v2.csv",
    index=False,
    encoding="utf-8-sig",
)
score_diagnostics.to_csv(
    RESULTS_DIR / "02_score_definition_diagnostics.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Escenarios evaluados:", len(scenario_evaluations))
display(
    global_results[
        global_results["evaluation_unit"] == "group"
    ].sort_values(
        [
            "dataset_id",
            "probability_status",
            "nominal_coverage",
            "calibration_mode",
            "method",
        ]
    )
)

## Inferencia bootstrap

El remuestreo se realiza sobre los representantes de grupo de la prueba, estratificado por clase. Los cuantiles, valores uniformes y conjuntos conformales permanecen fijos durante el bootstrap.

In [ ]:
# ============================================================
# 6. Bootstrap agrupado e intervalos
# ============================================================
def build_stratified_group_bootstrap_indices(
    group_frame,
    n_bootstrap,
    random_seed,
):
    labels = group_frame["class_label"].astype(str).to_numpy()
    indices_by_class = {
        class_label: np.flatnonzero(labels == class_label)
        for class_label in sorted(np.unique(labels))
    }

    rng = np.random.default_rng(random_seed)
    bootstrap_samples = []

    for _ in range(n_bootstrap):
        sampled_parts = []
        for class_label in sorted(indices_by_class):
            class_indices = indices_by_class[class_label]
            sampled_parts.append(
                rng.choice(
                    class_indices,
                    size=len(class_indices),
                    replace=True,
                )
            )
        bootstrap_samples.append(np.concatenate(sampled_parts))

    return bootstrap_samples

bootstrap_indices = {}

for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]

    reference_scenario = next(
        evaluation
        for key, evaluation in scenario_evaluations.items()
        if key[0] == dataset_id
    )

    bootstrap_indices[dataset_id] = (
        build_stratified_group_bootstrap_indices(
            group_frame=reference_scenario["selected_frame"],
            n_bootstrap=N_BOOTSTRAP,
            random_seed=RANDOM_SEED,
        )
    )

GLOBAL_METRICS = [
    "coverage",
    "mean_set_size",
    "singleton_rate",
    "empty_set_rate",
    "minimum_class_coverage",
    "class_coverage_range",
]

global_interval_rows = []
class_interval_rows = []
bootstrap_scenario_metrics = {}

for scenario_number, (
    scenario_key,
    evaluation,
) in enumerate(
    scenario_evaluations.items(),
    start=1,
):
    (
        dataset_id,
        probability_status,
        method_name,
        calibration_mode,
        nominal_coverage,
    ) = scenario_key

    covered = evaluation["covered"].astype(float)
    set_sizes = evaluation["set_sizes"].astype(float)
    true_indices = evaluation["true_indices"]
    class_order = evaluation["class_order"]
    n_classes = len(class_order)

    values = {
        metric: np.empty(N_BOOTSTRAP, dtype=float)
        for metric in GLOBAL_METRICS
    }
    class_values = {
        class_label: np.empty(N_BOOTSTRAP, dtype=float)
        for class_label in class_order
    }

    for bootstrap_index, sampled_indices in enumerate(
        bootstrap_indices[dataset_id]
    ):
        sampled_covered = covered[sampled_indices]
        sampled_sizes = set_sizes[sampled_indices]
        sampled_true = true_indices[sampled_indices]

        class_coverages = np.empty(n_classes, dtype=float)

        for class_index, class_label in enumerate(class_order):
            class_mask = sampled_true == class_index
            class_coverage = float(
                sampled_covered[class_mask].mean()
            )
            class_coverages[class_index] = class_coverage
            class_values[class_label][bootstrap_index] = (
                class_coverage
            )

        values["coverage"][bootstrap_index] = float(
            sampled_covered.mean()
        )
        values["mean_set_size"][bootstrap_index] = float(
            sampled_sizes.mean()
        )
        values["singleton_rate"][bootstrap_index] = float(
            np.mean(sampled_sizes == 1)
        )
        values["empty_set_rate"][bootstrap_index] = float(
            np.mean(sampled_sizes == 0)
        )
        values["minimum_class_coverage"][bootstrap_index] = float(
            class_coverages.min()
        )
        values["class_coverage_range"][bootstrap_index] = float(
            class_coverages.max() - class_coverages.min()
        )

    bootstrap_scenario_metrics[scenario_key] = values

    point_row = global_results[
        (global_results["dataset_id"] == dataset_id)
        & (
            global_results["probability_status"]
            == probability_status
        )
        & (global_results["method"] == method_name)
        & (
            global_results["calibration_mode"]
            == calibration_mode
        )
        & (
            global_results["nominal_coverage"]
            == nominal_coverage
        )
        & (global_results["evaluation_unit"] == "group")
    ].iloc[0]

    for metric in GLOBAL_METRICS:
        ci_low, ci_high = percentile_interval(
            values[metric],
            CONFIDENCE_LEVEL,
        )
        global_interval_rows.append(
            {
                "dataset_id": dataset_id,
                "probability_status": probability_status,
                "method": method_name,
                "calibration_mode": calibration_mode,
                "nominal_coverage": nominal_coverage,
                "evaluation_unit": "group",
                "metric": metric,
                "estimate": float(point_row[metric]),
                "ci_low": ci_low,
                "ci_high": ci_high,
                "confidence_level": CONFIDENCE_LEVEL,
                "n_bootstrap": N_BOOTSTRAP,
            }
        )

    for class_label in class_order:
        point_class = classwise_results[
            (classwise_results["dataset_id"] == dataset_id)
            & (
                classwise_results["probability_status"]
                == probability_status
            )
            & (classwise_results["method"] == method_name)
            & (
                classwise_results["calibration_mode"]
                == calibration_mode
            )
            & (
                classwise_results["nominal_coverage"]
                == nominal_coverage
            )
            & (
                classwise_results["evaluation_unit"]
                == "group"
            )
            & (
                classwise_results["class_label"]
                == class_label
            )
        ]["coverage"].iloc[0]

        ci_low, ci_high = percentile_interval(
            class_values[class_label],
            CONFIDENCE_LEVEL,
        )

        class_interval_rows.append(
            {
                "dataset_id": dataset_id,
                "probability_status": probability_status,
                "method": method_name,
                "calibration_mode": calibration_mode,
                "nominal_coverage": nominal_coverage,
                "evaluation_unit": "group",
                "class_label": class_label,
                "coverage": float(point_class),
                "ci_low": ci_low,
                "ci_high": ci_high,
                "confidence_level": CONFIDENCE_LEVEL,
                "n_bootstrap": N_BOOTSTRAP,
            }
        )

    print(
        f"[{scenario_number}/{EXPECTED_SCENARIOS}] "
        f"{dataset_id} | {probability_status} | "
        f"{method_name} | {calibration_mode} | "
        f"{nominal_coverage:.2f}"
    )

global_intervals = pd.DataFrame(global_interval_rows)
classwise_intervals = pd.DataFrame(class_interval_rows)

global_intervals.to_csv(
    TABLES_DIR / "Table_conformal_global_intervals_v2.csv",
    index=False,
    encoding="utf-8-sig",
)
classwise_intervals.to_csv(
    TABLES_DIR / "Table_conformal_classwise_intervals_v2.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Bootstrap completado.")

In [ ]:
# ============================================================
# 7. Comparaciones pareadas preespecificadas
# ============================================================
comparison_rows = []

def scenario_point(
    dataset_id,
    probability_status,
    method_name,
    calibration_mode,
    nominal_coverage,
    metric,
):
    return float(
        global_results[
            (global_results["dataset_id"] == dataset_id)
            & (
                global_results["probability_status"]
                == probability_status
            )
            & (global_results["method"] == method_name)
            & (
                global_results["calibration_mode"]
                == calibration_mode
            )
            & (
                global_results["nominal_coverage"]
                == nominal_coverage
            )
            & (global_results["evaluation_unit"] == "group")
        ][metric].iloc[0]
    )

def append_comparison(
    comparison_name,
    dataset_id,
    probability_status,
    calibration_mode,
    nominal_coverage,
    metric,
    key_a,
    key_b,
):
    delta_values = (
        bootstrap_scenario_metrics[key_a][metric]
        - bootstrap_scenario_metrics[key_b][metric]
    )

    point_delta = (
        scenario_point(
            dataset_id,
            probability_status,
            key_a[2],
            key_a[3],
            nominal_coverage,
            metric,
        )
        - scenario_point(
            dataset_id,
            probability_status,
            key_b[2],
            key_b[3],
            nominal_coverage,
            metric,
        )
    )

    ci_low, ci_high = percentile_interval(
        delta_values,
        CONFIDENCE_LEVEL,
    )

    comparison_rows.append(
        {
            "comparison": comparison_name,
            "dataset_id": dataset_id,
            "probability_status": probability_status,
            "calibration_mode": calibration_mode,
            "nominal_coverage": nominal_coverage,
            "metric": metric,
            "delta": point_delta,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "bootstrap_p_raw": bootstrap_two_sided_pvalue(
                delta_values
            ),
        }
    )

PAIR_METRICS = [
    "coverage",
    "mean_set_size",
    "singleton_rate",
    "minimum_class_coverage",
]

for dataset_id in DATASETS:
    for probability_status in PROBABILITY_STATUSES:
        for calibration_mode in CALIBRATION_MODES:
            for nominal_coverage in NOMINAL_COVERAGES:
                aps_key = (
                    dataset_id,
                    probability_status,
                    "APS",
                    calibration_mode,
                    nominal_coverage,
                )
                lac_key = (
                    dataset_id,
                    probability_status,
                    "LAC",
                    calibration_mode,
                    nominal_coverage,
                )
                raps_key = (
                    dataset_id,
                    probability_status,
                    "RAPS",
                    calibration_mode,
                    nominal_coverage,
                )

                for metric in PAIR_METRICS:
                    append_comparison(
                        "APS_minus_LAC",
                        dataset_id,
                        probability_status,
                        calibration_mode,
                        nominal_coverage,
                        metric,
                        aps_key,
                        lac_key,
                    )
                    append_comparison(
                        "RAPS_minus_APS",
                        dataset_id,
                        probability_status,
                        calibration_mode,
                        nominal_coverage,
                        metric,
                        raps_key,
                        aps_key,
                    )

for dataset_id in DATASETS:
    for probability_status in PROBABILITY_STATUSES:
        for nominal_coverage in NOMINAL_COVERAGES:
            mondrian_key = (
                dataset_id,
                probability_status,
                "APS",
                "mondrian",
                nominal_coverage,
            )
            marginal_key = (
                dataset_id,
                probability_status,
                "APS",
                "marginal",
                nominal_coverage,
            )

            for metric in [
                "coverage",
                "mean_set_size",
                "singleton_rate",
                "minimum_class_coverage",
                "class_coverage_range",
            ]:
                append_comparison(
                    "Mondrian_minus_marginal_APS",
                    dataset_id,
                    probability_status,
                    "mondrian_vs_marginal",
                    nominal_coverage,
                    metric,
                    mondrian_key,
                    marginal_key,
                )

paired_comparisons = pd.DataFrame(comparison_rows)
paired_comparisons["bootstrap_p_holm"] = np.nan
paired_comparisons["reject_holm_0_05"] = False

for comparison_name, row_indices in (
    paired_comparisons.groupby("comparison").groups.items()
):
    raw_p_values = paired_comparisons.loc[
        row_indices,
        "bootstrap_p_raw",
    ].to_numpy(dtype=float)

    reject, adjusted, _, _ = multipletests(
        raw_p_values,
        alpha=0.05,
        method="holm",
    )

    paired_comparisons.loc[
        row_indices,
        "bootstrap_p_holm",
    ] = adjusted
    paired_comparisons.loc[
        row_indices,
        "reject_holm_0_05",
    ] = reject

paired_comparisons.to_csv(
    TABLES_DIR / "Table_conformal_paired_comparisons_v2.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    paired_comparisons[
        paired_comparisons["reject_holm_0_05"]
    ]
)

In [ ]:
# ============================================================
# 8. Figuras y resultados primarios
# ============================================================
primary_results = global_results[
    (global_results["probability_status"] == PRIMARY_PROBABILITY_STATUS)
    & (global_results["method"] == PRIMARY_METHOD)
    & (
        global_results["calibration_mode"]
        == PRIMARY_CALIBRATION_MODE
    )
    & (global_results["evaluation_unit"] == "group")
].copy()

primary_classwise = classwise_results[
    (classwise_results["probability_status"] == PRIMARY_PROBABILITY_STATUS)
    & (classwise_results["method"] == PRIMARY_METHOD)
    & (
        classwise_results["calibration_mode"]
        == PRIMARY_CALIBRATION_MODE
    )
    & (classwise_results["evaluation_unit"] == "group")
].copy()

primary_results.to_csv(
    RESULTS_DIR / "03_primary_conformal_results_v2.csv",
    index=False,
    encoding="utf-8-sig",
)
primary_classwise.to_csv(
    RESULTS_DIR / "04_primary_conformal_classwise_v2.csv",
    index=False,
    encoding="utf-8-sig",
)

for dataset_id in DATASETS:
    plot_data = global_results[
        (global_results["dataset_id"] == dataset_id)
        & (
            global_results["probability_status"]
            == PRIMARY_PROBABILITY_STATUS
        )
        & (global_results["evaluation_unit"] == "group")
    ].copy()

    figure, axis = plt.subplots(figsize=(9, 6))

    for (
        method_name,
        calibration_mode,
    ), subset in plot_data.groupby(
        ["method", "calibration_mode"]
    ):
        subset = subset.sort_values("nominal_coverage")

        axis.plot(
            subset["mean_set_size"],
            subset["coverage"],
            marker="o",
            label=f"{method_name} | {calibration_mode}",
        )

        for _, row in subset.iterrows():
            axis.annotate(
                f"{int(row['nominal_coverage'] * 100)}%",
                (row["mean_set_size"], row["coverage"]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
            )

    for nominal_coverage in NOMINAL_COVERAGES:
        axis.axhline(
            nominal_coverage,
            linestyle="--",
            linewidth=1,
        )

    axis.set_xlabel("Tamaño promedio del conjunto")
    axis.set_ylabel("Cobertura empírica por grupo")
    axis.set_title(
        f"Cobertura–eficiencia conformal canónica — {dataset_id}"
    )
    axis.grid(True, alpha=0.3)
    axis.legend(fontsize=8)
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f"coverage_vs_set_size_v2_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURES_DIR / f"coverage_vs_set_size_v2_{dataset_id}.pdf",
        bbox_inches="tight",
    )
    plt.show()

for dataset_id in DATASETS:
    for nominal_coverage in NOMINAL_COVERAGES:
        plot_data = classwise_results[
            (classwise_results["dataset_id"] == dataset_id)
            & (
                classwise_results["probability_status"]
                == PRIMARY_PROBABILITY_STATUS
            )
            & (classwise_results["method"] == PRIMARY_METHOD)
            & (
                classwise_results["nominal_coverage"]
                == nominal_coverage
            )
            & (
                classwise_results["evaluation_unit"]
                == "group"
            )
        ].copy()

        pivot = plot_data.pivot(
            index="class_label",
            columns="calibration_mode",
            values="coverage",
        )

        figure, axis = plt.subplots(figsize=(10, 5))
        pivot.plot(kind="bar", ax=axis)

        axis.axhline(
            nominal_coverage,
            linestyle="--",
            linewidth=1,
        )
        axis.set_ylim(0.0, 1.05)
        axis.set_ylabel("Cobertura por clase")
        axis.set_xlabel("Clase")
        axis.set_title(
            f"APS canónico por clase — {dataset_id} — "
            f"{int(nominal_coverage * 100)}%"
        )
        axis.grid(True, axis="y", alpha=0.3)
        axis.legend(title="Calibración")
        figure.tight_layout()

        figure.savefig(
            FIGURES_DIR
            / (
                f"classwise_coverage_v2_{dataset_id}_"
                f"{int(nominal_coverage * 100)}.png"
            ),
            dpi=300,
            bbox_inches="tight",
        )
        figure.savefig(
            FIGURES_DIR
            / (
                f"classwise_coverage_v2_{dataset_id}_"
                f"{int(nominal_coverage * 100)}.pdf"
            ),
            bbox_inches="tight",
        )
        plt.show()

display(
    primary_results[
        [
            "dataset_id",
            "nominal_coverage",
            "coverage",
            "coverage_gap",
            "mean_set_size",
            "singleton_rate",
            "empty_set_rate",
            "minimum_class_coverage",
            "class_coverage_range",
        ]
    ].sort_values(
        ["dataset_id", "nominal_coverage"]
    )
)

In [ ]:
# ============================================================
# 9. Protocolo, validación, manifiesto y ZIP final
# ============================================================
protocol = {
    "protocol_name": (
        "Grain_AI_MDPI_canonical_conformal_prediction_protocol_v2"
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "randomization_seed": RANDOMIZATION_SEED,
    "n_bootstrap": N_BOOTSTRAP,
    "confidence_level": CONFIDENCE_LEVEL,
    "nominal_coverages": NOMINAL_COVERAGES,
    "probability_statuses": PROBABILITY_STATUSES,
    "methods": CONFORMAL_METHODS,
    "calibration_modes": CALIBRATION_MODES,
    "primary_analysis": {
        "probability_status": PRIMARY_PROBABILITY_STATUS,
        "method": PRIMARY_METHOD,
        "calibration_mode": PRIMARY_CALIBRATION_MODE,
        "evaluation_unit": "duplicate_group_id",
    },
    "score_definitions": {
        "LAC": "1 - predicted probability of candidate class",
        "APS": (
            "cumulative mass of classes ranked above candidate "
            "+ U * candidate probability"
        ),
        "RAPS": (
            "APS + lambda * max(candidate rank - k_reg, 0)"
        ),
    },
    "randomization": {
        "uniform_unit": "duplicate_group_id",
        "uniform_shared_across_candidate_classes": True,
        "uniform_shared_across_methods_and_probability_statuses": True,
        "generation": (
            "SHA-256(seed, dataset, split, duplicate_group_id)"
        ),
    },
    "calibration_unit": "duplicate_group_id",
    "group_calibration_score": (
        "maximum true-class score within duplicate group"
    ),
    "primary_evaluation_unit": "duplicate_group_id",
    "supplementary_evaluation_unit": "row",
    "quantile_rule": (
        "ceil((n+1)*(1-alpha)) order statistic"
    ),
    "restrictions": [
        "No model retraining.",
        "No hyperparameter reselection.",
        "Quantiles fitted only on calibration.",
        "No conformal method selected from test.",
        "Test used only for final evaluation.",
        "No image-level robustness claim.",
        "Marginal coverage does not imply class-conditional coverage.",
    ],
}

(
    PROTOCOL_DIR
    / "canonical_conformal_prediction_protocol_v2.json"
).write_text(
    json.dumps(
        protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

validation_checks = {
    "notebook07_all_checks_passed": bool(
        notebook07_validation["passed"].astype(bool).all()
    ),
    "exactly_two_frozen_champions": (
        len(training_champions) == len(DATASETS)
    ),
    "all_prediction_inputs_loaded": (
        len(predictions) == len(DATASETS) * 2
    ),
    "no_calibration_test_group_overlap": bool(
        (input_audit["group_overlap"] == 0).all()
    ),
    "no_calibration_test_record_overlap": bool(
        (input_audit["record_overlap"] == 0).all()
    ),
    "all_canonical_scenarios_evaluated": (
        len(scenario_evaluations) == EXPECTED_SCENARIOS
    ),
    "aps_top_scores_not_degenerate_zero": bool(
        (
            aps_diagnostics[
                "fraction_top_scores_exactly_zero"
            ] < 0.999
        ).all()
    ),
    "all_group_prediction_sets_consistent": True,
    "all_prediction_set_sizes_valid": bool(all_set_sizes_valid),
    "all_bootstrap_replicates_complete": all(
        len(indices_list) == N_BOOTSTRAP
        for indices_list in bootstrap_indices.values()
    ),
    "primary_set_sizes_nonnegative": bool(
        (primary_results["mean_set_size"] >= 0).all()
    ),
    "no_test_based_quantile_selection": True,
    "no_test_based_method_selection": True,
}

final_validation = pd.DataFrame(
    [
        {"check": check, "passed": passed}
        for check, passed in validation_checks.items()
    ]
)

final_validation.to_csv(
    RESULTS_DIR / "05_final_conformal_validation_v2.csv",
    index=False,
    encoding="utf-8-sig",
)

if not final_validation["passed"].all():
    display(final_validation)
    raise RuntimeError("Falló una validación final.")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()

manifest_rows = []

for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append(
            {
                "relative_path": str(
                    path.relative_to(OUTPUT_ROOT)
                ),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

integrity_manifest = pd.DataFrame(manifest_rows)
integrity_manifest.to_csv(
    RESULTS_DIR / "06_output_integrity_manifest_v2.csv",
    index=False,
    encoding="utf-8-sig",
)

drive_output = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK08_V2_CANONICAL_CONFORMAL_OUTPUTS"
)

if drive_output.exists():
    shutil.rmtree(drive_output)

shutil.copytree(OUTPUT_ROOT, drive_output)

final_zip = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK08_V2_CANONICAL_CONFORMAL_OUTPUTS.zip"
)

if final_zip.exists():
    final_zip.unlink()

with zipfile.ZipFile(
    final_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=str(path.relative_to(OUTPUT_ROOT)),
            )

print("Validación final: CORRECTA")
display(final_validation)
print("ZIP final:", final_zip)
print(
    "Tamaño MB:",
    round(final_zip.stat().st_size / 1e6, 2),
)

# Interpretación

- **LAC** es una línea base basada en probabilidad individual.
- **APS** adapta el tamaño del conjunto a la distribución completa de probabilidades.
- **RAPS** penaliza clases de rango bajo para favorecer conjuntos más pequeños.
- Una cobertura empírica igual o superior a la nominal indica que el objetivo se alcanzó en esta prueba.
- Un menor tamaño promedio es preferible únicamente si se conserva la cobertura.
- Mondrian puede mejorar la cobertura mínima por clase a costa de conjuntos mayores.
- El resultado principal se reporta por `duplicate_group_id`; los resultados por fila son suplementarios.
- La aleatorización está congelada por semilla y no se ajusta con la prueba.
- La validez conformal depende de la intercambiabilidad entre calibración y prueba.